# Get bounding boxes as ground truth for explanations

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import matplotlib.pyplot as plt
import decord as de
import random
from mmaction.apis import inference_recognizer, init_recognizer
from skimage.color import gray2rgb
from scipy import ndimage
import cv2
from mmcv import Config
import csv
import numpy as np
import pickle
from ultralytics import YOLO
import torch

sys.path.insert(0, os.path.join(sys.path[0], '../video_xai'))
from evaluators import *
from utils import *

d:\ProgramData\Anaconda3\envs\open-mmlab\lib\site-packages\mmcv\cnn\bricks\transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '


## UCF101

Corrected labels for UCF101-24, downloaded from: https://github.com/gurkirt/corrected-UCF101-Annots/tree/master.

In [32]:
ann_path = "Z:/Xavi/UCF101/corrected-UCF101-Annots-master/pyannot.pkl"
explanations_root = "Z:/Xavi/VideoXAI/batch_explanations/UCF101"
videos_path = os.path.join(explanations_root, "videos")
boxes_path = os.path.join(explanations_root, "boxes")

# Create folder
if not os.path.exists(boxes_path):
    os.makedirs(boxes_path)

# Load UCF101 annotations
with open(ann_path, 'rb') as f:
    data_aux = pickle.load(f)

# Fix keys
data = {}
for key in data_aux.keys():
    if len(key.split('/')) > 1:
        data[key.split('/')[1]] = data_aux[key]
    else:
        print(key)

# Iterate over videos
for label in os.listdir(videos_path):

    # Create folder
    if not os.path.exists(os.path.join(boxes_path, label)):
        os.makedirs(os.path.join(boxes_path, label))

    for video_name in os.listdir(os.path.join(videos_path, label)):
        print(f"Processing {label}/{video_name}")
        
        # Load video, get annotation and initialize mask
        video = load_video(os.path.join(videos_path, label, video_name))
        ann = data[video_name.split('.')[0]]
        mask = np.zeros(video.shape[:3], dtype=np.uint8)

        # Create mask
        for boxes_list in ann['annotations']:
            frame_i = boxes_list['sf']
            for box in boxes_list['boxes']:
                mask[frame_i:frame_i+1 , box[1]:box[1]+box[3], box[0]:box[0]+box[2]] = 1
                frame_i += 1

        # Resize and crop mask
        scale_factor = 256 / min(video.shape[1:3])
        mask = resize_video(mask, scale_factor, scale_factor, cv2.INTER_NEAREST)
        mask = center_crop_video(mask, 256)

        # Save mask
        np.savez_compressed(os.path.join(boxes_path, label, video_name), box=mask)

Processing 10/v_Biking_g08_c04.avi
Processing 10/v_Biking_g20_c01.avi


d:\ProgramData\Anaconda3\envs\open-mmlab\lib\site-packages\ipykernel_launcher.py:41: RuntimeWarning: overflow encountered in ubyte_scalars


Processing 21/v_CliffDiving_g06_c01.avi
Processing 21/v_CliffDiving_g20_c04.avi
Processing 21/v_CliffDiving_g20_c05.avi
Processing 22/v_CricketBowling_g19_c05.avi
Processing 25/v_Diving_g20_c03.avi
Processing 25/v_Diving_g22_c04.avi
Processing 27/v_Fencing_g04_c04.avi
Processing 32/v_GolfSwing_g17_c04.avi
Processing 43/v_IceDancing_g02_c07.avi
Processing 43/v_IceDancing_g12_c03.avi
Processing 50/v_LongJump_g02_c05.avi
Processing 50/v_LongJump_g16_c01.avi
Processing 67/v_PoleVault_g25_c07.avi
Processing 7/v_Basketball_g08_c04.avi
Processing 7/v_Basketball_g23_c04.avi
Processing 74/v_RopeClimbing_g10_c01.avi
Processing 76/v_SalsaSpin_g06_c01.avi
Processing 76/v_SalsaSpin_g07_c01.avi
Processing 8/v_BasketballDunk_g11_c04.avi
Processing 8/v_BasketballDunk_g14_c01.avi
Processing 80/v_Skiing_g24_c03.avi
Processing 81/v_Skijet_g14_c01.avi
Processing 81/v_Skijet_g25_c03.avi
Processing 91/v_TennisSwing_g06_c03.avi
Processing 91/v_TennisSwing_g06_c06.avi
Processing 91/v_TennisSwing_g23_c06.avi
P

## EtriActivity3D and Kinetics400

Labels computed using YOLOv8

In [5]:
dataset = "Kinetics400"

explanations_root = "Z:/Xavi/VideoXAI/batch_explanations/"+dataset
videos_path = os.path.join(explanations_root, "videos")
boxes_path = os.path.join(explanations_root, "boxes")

# Initialize YOLO
weights_path = 'C:/Users/Xavi/OneDrive - Universitat de les Illes Balears/Doctorat/Computer Vision Module/robot_vision/robot_vision/models/yolov8x_person_face.pt'
yolo = YOLO(weights_path)

# Iterate over videos
for label in os.listdir(videos_path):

    # Create folder
    if not os.path.exists(os.path.join(boxes_path, label)):
        os.makedirs(os.path.join(boxes_path, label))

    for video_name in os.listdir(os.path.join(videos_path, label)):
        print(f"Processing {label}/{video_name}")
        
        # Load video and run YOLO
        video = load_video(os.path.join(videos_path, label, video_name))

        # Run YOLO
        boxes = []
        for frame in video:
            boxes.append(yolo([frame], verbose=False, classes=[0])[0])

            # Clear memory
            torch.cuda.empty_cache()

        # Create mask
        mask = np.zeros(video.shape[:3], dtype=np.uint8)
        for frame_i, frame_boxes in enumerate(boxes):
            for box in frame_boxes.boxes.xyxy:
                box = box.int()
                mask[frame_i:frame_i+1 , box[1]:box[3], box[0]:box[2]] = 1
        
        # Resize and crop mask
        scale_factor = 256 / min(video.shape[1:3])
        mask = resize_video(mask, scale_factor, scale_factor, cv2.INTER_NEAREST)
        mask = center_crop_video(mask, 256)

        # Save mask
        np.savez_compressed(os.path.join(boxes_path, label, video_name), box=mask)

Processing 101/--6bJUbfpnQ_000017_000027.mp4
Processing 135/Y8BCvE5Ai1o_000002_000012.mp4
Processing 147/Cc5R3Ey3MQw_000004_000014.mp4
Processing 15/T6seaVGSuEY_000048_000058.mp4
Processing 155/JYItC7ahu_s_000215_000225.mp4
Processing 165/DJSqNV9m-2A_000019_000029.mp4
Processing 176/6mvqsdJcazA_000018_000028.mp4
Processing 177/S5C0fUV6yEo_000003_000013.mp4
Processing 183/OVcjZPMeDDc_000001_000011.mp4
Processing 213/fcD17GSzCI8_000001_000011.mp4
Processing 214/mY0Zh6IYpgA_000045_000055.mp4
Processing 220/FOV6Yqg6B_Q_000004_000014.mp4
Processing 231/2ZP4Tcq4nq8_000020_000030.mp4
Processing 236/2G1sRvmbza0_000000_000010.mp4
Processing 243/E-KmvjxUsd8_000110_000120.mp4
Processing 244/0q1c6APn11c_000043_000053.mp4
Processing 252/vmlKvajfs_I_000191_000201.mp4
Processing 253/LuBHCaA0bo4_000020_000030.mp4
Processing 256/xQ5PoSoQnK0_000073_000083.mp4
Processing 298/72vAM2L477g_000015_000025.mp4
Processing 301/WLlSvm6yuj4_000011_000021.mp4
Processing 32/gM4HJUB1AlA_000143_000153.mp4
Processing 3

# Visualize bounding boxes

In [6]:
dataset = 'Kinetics400'
explanations_root = "Z:/Xavi/VideoXAI/batch_explanations/"+dataset
videos_path = os.path.join(explanations_root, "videos_small")
boxes_path = os.path.join(explanations_root, "boxes")

boxes_on_video_path = os.path.join(explanations_root, "videos_with_boxes")

# Create folder
if not os.path.exists(boxes_on_video_path):
    os.makedirs(boxes_on_video_path)

# Iterate over videos
for label in os.listdir(videos_path):

    # Create folder
    if not os.path.exists(os.path.join(boxes_on_video_path, label)):
        os.makedirs(os.path.join(boxes_on_video_path, label))

    for video_name in os.listdir(os.path.join(videos_path, label)):
        print(f"Processing {label}/{video_name}")
        
        # Load video and mask
        video = load_video(os.path.join(videos_path, label, video_name))
        with np.load(os.path.join(boxes_path, label, video_name+'.npz')) as data:
            mask = data['box']

        # Add mask to video
        video_with_boxes = painter(video, mask[..., None]*255)

        # Save video with boxes
        save_video(video_with_boxes, os.path.join(boxes_on_video_path, label, video_name))

Processing 101/--6bJUbfpnQ_000017_000027.mp4
Processing 135/Y8BCvE5Ai1o_000002_000012.mp4
Processing 147/Cc5R3Ey3MQw_000004_000014.mp4
Processing 15/T6seaVGSuEY_000048_000058.mp4
Processing 155/JYItC7ahu_s_000215_000225.mp4
Processing 165/DJSqNV9m-2A_000019_000029.mp4
Processing 176/6mvqsdJcazA_000018_000028.mp4
Processing 177/S5C0fUV6yEo_000003_000013.mp4
Processing 183/OVcjZPMeDDc_000001_000011.mp4
Processing 213/fcD17GSzCI8_000001_000011.mp4
Processing 214/mY0Zh6IYpgA_000045_000055.mp4
Processing 220/FOV6Yqg6B_Q_000004_000014.mp4
Processing 231/2ZP4Tcq4nq8_000020_000030.mp4
Processing 236/2G1sRvmbza0_000000_000010.mp4
Processing 243/E-KmvjxUsd8_000110_000120.mp4
Processing 244/0q1c6APn11c_000043_000053.mp4
Processing 252/vmlKvajfs_I_000191_000201.mp4
Processing 253/LuBHCaA0bo4_000020_000030.mp4
Processing 256/xQ5PoSoQnK0_000073_000083.mp4
Processing 298/72vAM2L477g_000015_000025.mp4
Processing 301/WLlSvm6yuj4_000011_000021.mp4
Processing 32/gM4HJUB1AlA_000143_000153.mp4
Processing 3